# Preparation of final ESCO-KldB Crosswalk
Felix Zaussinger | 16.08.2022

#### Summary
- Final crosswalk maps directly to the KldB (previous version mapped to Berufenet)
- 1207 unique KldB occupations are represented, compared to 920 before
- 93 matches are between ISCO occupation groups

In [2]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils, plotting_utils
#import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("talk")
sns.set(rc={'figure.figsize': (6, 3.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

In [3]:
def print_uniques_per_col(df):
    for col in df.columns:
        print(col, " : ", df[col].unique().shape[0])

#### Prepare ESCO - KldB crosswalk

Two steps:
1. ESCO - Berufenet DKZ ID (3-6 digits)
2. DKZ ID - KldB 2010 code (5 digits)

In [9]:
fpath_final_cw = os.path.join(
    useful_paths.data_raw,
    "crosswalks",
    "Mapping_ESCO_KldB2010_220518.xlsx"
)
cw_kldb = pd.read_excel(fpath_final_cw, header=16)
cw_kldb.columns = cw_kldb.columns.str.split(
                " "
            ).str.join("_")
cw_kldb

,Classification_1_URI,Classification_1_PrefLabel,Classification_2_ID_Raw,Classification_2_ID,Classification_2_PrefLabel,Mapping_relation,Editorial_note
0,http://data.europa.eu/esco/isco/C0110,Offiziere in regulären Streitkräften,B 01104-104,1104,Offizier im Sanitätsdienst,skos:narrowMatch,Alternativmapping über ISCO-2008
1,http://data.europa.eu/esco/isco/C0110,Offiziere in regulären Streitkräften,B 01104-105,1104,Offizier im Geoinformationsdienst,skos:narrowMatch,Alternativmapping über ISCO-2008
2,http://data.europa.eu/esco/isco/C0110,Offiziere in regulären Streitkräften,B 01104-106,1104,Offizier im Militärmusikdienst,skos:narrowMatch,Alternativmapping über ISCO-2008
3,http://data.europa.eu/esco/isco/C0210,Unteroffiziere in regulären Streitkräften,B 01302-101,1302,Fachunteroffizier im allgemeinen Fachdienst,skos:narrowMatch,Alternativmapping über ISCO-2008
4,http://data.europa.eu/esco/isco/C0210,Unteroffiziere in regulären Streitkräften,B 01203-100,1203,Feldwebel im Truppendienst,skos:narrowMatch,Alternativmapping über ISCO-2008
...,...,...,...,...,...,...,...
8730,http://data.europa.eu/esco/occupation/e16e4cb0...,Waschsalongehilfe/Waschsalongehilfin,BE 9629,9629,"Hilfsarbeitskräfte, anderweitig nicht genannt",skos:broadMatch,Alternativmapping über ISCO-2008
8731,http://data.europa.eu/esco/occupation/9e4269d5...,Zentralbankgouverneur/Zentralbankgouverneurin,BE 1112,1112,Leitende Verwaltungsbedienstete,skos:broadMatch,Alternativmapping über ISCO-2008
8732,http://data.europa.eu/esco/occupation/6e1b1bd7...,Zulassungskoordinator/Zulassungskoordinatorin,BE 2359,2359,"Lehrkräfte, anderweitig nicht genannt",skos:broadMatch,Alternativmapping über ISCO-2008
8733,http://data.europa.eu/esco/occupation/39c89133...,Zulassungssachbearbeiter/Zulassungssachbearbei...,BE 3354,3354,"Fachkräfte bei staatlichen Pass-, Lizenz- und ...",skos:broadMatch,Alternativmapping über ISCO-2008


In [10]:
unique_occs = cw_kldb.Classification_2_ID.dropna().unique()
len_of_occ_codes = pd.Series(unique_occs).astype(int).astype(str).str.len()
np.unique(len_of_occ_codes, return_counts=True)

(array([3, 4, 5], dtype=int64), array([   3,   89, 1207], dtype=int64))

In [11]:
print_uniques_per_col(cw_kldb)

Classification_1_URI  :  2969
Classification_1_PrefLabel  :  2968
Classification_2_ID_Raw  :  4377
Classification_2_ID  :  1299
Classification_2_PrefLabel  :  4377
Mapping_relation  :  4
Editorial_note  :  2


Kldb 2010 names

In [14]:
kldb_names = pd.read_excel(r"T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\rcode\B_Metadaten\Kldb2010-Englisch.xls", sheet_name="OccupationMetadata", usecols=["kldb2010_code", "kldb2010_name_de", "kldb2010_name_en"])
kldb_names

,kldb2010_code,kldb2010_name_de,kldb2010_name_en
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...
1,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...
2,11103,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...
3,11104,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...
4,11113,Berufe in der Landtechnik - komplexe Spezialis...,Technical occupations in farming-complex tasks
...,...,...,...
1281,94794,Führungskräfte im Museum,Managers in museum
1282,1104,Offiziere,Commissioned officers
1283,1203,Unteroffiziere mit Portepee,Senior non-commissioned officers and higher
1284,1302,Unteroffiziere ohne Portepee,Junior non-commissioned officers


Combine all data and save

In [18]:
cw_kldb_merged_final = pd.merge(cw_kldb, kldb_names, left_on="Classification_2_ID", right_on="kldb2010_code", how="left")
print_uniques_per_col(cw_kldb_merged_final)

Classification_1_URI  :  2969
Classification_1_PrefLabel  :  2968
Classification_2_ID_Raw  :  4377
Classification_2_ID  :  1299
Classification_2_PrefLabel  :  4377
Mapping_relation  :  4
Editorial_note  :  2
kldb2010_code  :  1198
kldb2010_name_de  :  1198
kldb2010_name_en  :  1197


In [23]:
print_uniques_per_col(cw_kldb_merged_final.dropna(subset=["kldb2010_code"]))

Classification_1_URI  :  2833
Classification_1_PrefLabel  :  2832
Classification_2_ID_Raw  :  4263
Classification_2_ID  :  1197
Classification_2_PrefLabel  :  4263
Mapping_relation  :  4
Editorial_note  :  2
kldb2010_code  :  1197
kldb2010_name_de  :  1197
kldb2010_name_en  :  1196


In [24]:
cw_kldb_merged_final.to_csv(os.path.join(useful_paths.data_processed, "crosswalks", "crosswalk_esco_kldb2010_final.csv"))
cw_kldb_merged_final.to_pickle(os.path.join(useful_paths.data_processed, "crosswalks", "crosswalk_esco_kldb2010_final.pkl"))